# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook demonstrates a step-by-step exploration and processing of the FAIR² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant) library.

### Dataset Source
The dataset is defined by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json), enabling programmatic metadata introspection and powerful interoperability.

In [ ]:
# Ensure the required library is installed
!pip install mlcroissant

## 1. Data Loading

We will load the dataset metadata and explore basic metadata fields available in the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata summary
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Description: {metadata.description}")
print(f"Date published: {metadata.datePublished}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview

Let's inspect the available record sets and their fields. Every entity is referenced by its `@id` field for consistency and interoperability.

Note: In Croissant, a *record set* represents a table or logical group of records (rows), and each *field* represents a column/attribute.

In [ ]:
# List all record sets by @id
print("Available Record Sets by @id:")
record_sets = dataset.record_sets  # List of mlcroissant.entities.RecordSet
for rs in record_sets:
    print(f"  - RecordSet @id: {rs.id}, name: {rs.name}")

# For each record set, list its fields and columns with their @id's
for rs in record_sets:
    print(f"\nFields in RecordSet '{rs.name}' (@id: {rs.id}):")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'dataType', None)}")
        # If the field has columns
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"        > Column @id: {col.id}, name: {col.name}, dataType: {getattr(col, 'dataType', None)}")

> **Note:**
Depending on the dataset, there may be one or more record sets (tables) available. Each set is identified by its `@id`. We will now fetch a preview of records for each record set.

Below, we print a small sample for each record set. Use the `@id` fields for precise referencing.

In [ ]:
# Sample few records using their record set @id
for rs in record_sets:
    print(f"\nPreview of records for RecordSet '{rs.name}' (@id: {rs.id}):")
    for i, rec in enumerate(dataset.records(record_set=rs.id)):
        if i >= 3:
            break
        print(rec)

## 3. Data Extraction

Let's load each record set fully as a pandas DataFrame. We will use and reference them by their `@id`.

In [ ]:
# Prepare DataFrames for all record sets
dfs = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dfs[rs.id] = df
        print(f"\nDataFrame columns for RecordSet '{rs.name}' (@id: {rs.id}):")
        print(df.columns.tolist())
        print(df.head(3))

# For demonstration, pick the first record set's id for analysis below.
if record_sets:
    main_recordset_id = record_sets[0].id
else:
    main_recordset_id = None

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate processing on numeric fields, such as filtering, normalization, and grouping. All field/column names are referenced by their `@id` as per Croissant best practices.

In [ ]:
import numpy as np

# Pick the main DataFrame to analyze
df = dfs[main_recordset_id]

# Let's try to auto-detect a numeric field using Croissant field metadata
main_recordset = next((rs for rs in record_sets if rs.id == main_recordset_id), None)
# Get fields that are numeric (Float or Integer)
numeric_fields = [f for f in main_recordset.fields if getattr(f, 'dataType', None) in ["Float", "Integer", "Number"]]

if numeric_fields:
    numeric_field = numeric_fields[0].id  # Use @id
    print(f"Analyzing numeric field: {numeric_field}")
    # For convenience, get the column name as in DataFrame
    column_name = numeric_field if numeric_field in df.columns else df.columns[0]
else:
    print("No numeric fields found. Using first column.")
    numeric_field = df.columns[0]
    column_name = numeric_field

# Filter: Show only records with value > threshold
threshold = 10
if np.issubdtype(df[column_name].dtype, np.number):
    filtered_df = df[df[column_name] > threshold].copy()
else:
    # Try to convert to numeric
    filtered_df = df[pd.to_numeric(df[column_name], errors='coerce') > threshold].copy()

print(f"\nFiltered records with {column_name} (field @id: {numeric_field}) > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{column_name}_normalized"] = (
    filtered_df[column_name] - filtered_df[column_name].mean()
) / filtered_df[column_name].std()
print(f"\nNormalized {column_name} for filtered records:")
print(filtered_df[[column_name, f"{column_name}_normalized"]].head())

# Try grouping by a categorical field (@id)
categorical_fields = [f for f in main_recordset.fields if getattr(f, 'dataType', None) == "Text"]
if categorical_fields:
    group_field = categorical_fields[0].id
    print(f"\nGrouping by field: {group_field}")
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[column_name].mean().reset_index()
        print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of our chosen numeric field, as well as how it varies by the grouping variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[column_name], kde=True, bins=10)
plt.title(f"Distribution of {column_name} (@id: {numeric_field})")
plt.xlabel(column_name)
plt.ylabel("Frequency")
plt.show()

# If grouping field and grouped_df are available, show comparison
if 'grouped_df' in locals():
    plt.figure(figsize=(7,4))
    sns.barplot(data=grouped_df, x=group_field, y=column_name)
    plt.title(f"Mean {column_name} grouped by {group_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load and process a Croissant-compliant biomedical dataset with `mlcroissant`, exploring its metadata, structure, and content using only `@id` references for robust, schema-aware analysis.

- The dataset exposes its tables (record sets) and columns (fields/columns) with stable identifiers, which ensures consistent referencing for downstream workflows.
- We inspected data, filtered and normalized numeric columns, and explored groupwise patterns programmatically.

**Next steps:** Apply custom domain insights, explore other fields, or integrate the Croissant loader with your ML pipeline for reproducible biomedical analytics.